# RTX 4090 MoE operator scaling

This notebook runs the router and fused-MoE scans through a crash-isolated runner. Each shape gets a fresh CUDA process, so an OOM or kernel failure is retained as one failed case instead of corrupting the notebook kernel.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'tests':
    ROOT = ROOT.parent
if str(ROOT / 'tests') not in sys.path:
    sys.path.insert(0, str(ROOT / 'tests'))
from moe_operator_bench_runner import MoeOperatorBenchmark

benchmark = MoeOperatorBenchmark(
    output_dir='artifacts/operator_benchmark/rtx4090_24gb',
    warmup_ms=50,
    rep_ms=200,
    case_timeout_seconds=1200,
)
benchmark.preflight()

## Full scan

`project_formal` reaches the real 112 × 512 local-token workload. `mixtral_7b` preserves the prior 4096/14336 industrial reference shape while keeping correctness and gradient copies inside 24 GB.

In [ ]:
summary = benchmark.run()
{
    'quality_gate_passed': summary['quality_gate_passed'],
    'completed_rows': len(summary['rows']),
    'case_failures': len(summary['case_failures']),
    'correctness_failures': len(summary['correctness_failures']),
    'output_dir': summary['output_dir'],
}

In [ ]:
benchmark.display(summary)